# Publication figures — Arrow-of-Time experiment

Static, **vector-PDF** figures for papers, posters, and slides — the
print-ready counterpart of the interactive `dashboard.html`.

Each figure is a function in `analysis/figures.py`; this notebook just
drives them and displays the result inline. PDFs are written under
`analysis/derived/figures/pub/` (gitignored — regeneratable; the figure
*code* is what's version-controlled).

**Prerequisite:** the analysis pipeline must have been run so the
derived TSVs exist. If `analysis/derived/per_source.tsv` is missing or
stale, run the pipeline first (from `explore.ipynb`, or on the command
line):

```
python -m analysis.load_all
python -m analysis.per_subject
python -m analysis.per_video
python -m analysis.per_source
```


In [ ]:
# Make the `analysis` package importable from the notebook's folder,
# then pull in the figures module.
import sys, pathlib
_root = pathlib.Path.cwd()
for _ in range(5):
    if (_root / 'analysis' / '__init__.py').is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("couldn't locate the analysis package — open the notebook from inside the repo")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

%matplotlib inline
from analysis import figures
from analysis.figures import PUB_DIR

PUB_DIR.mkdir(parents=True, exist_ok=True)
print('publication PDFs will be written to:', PUB_DIR)


## Figure 1 — Per-source arrow-of-time asymmetry

Forward-render vs backward-render identifiability, one point per source
video, plotted on equal (1:1) axes.

- **Solid diagonal** `y = x` — raw symmetry.
- **Dashed diagonal** `y = x − offset` — *bias-adjusted* symmetry. The
  population forward-response bias inflates forward-render and deflates
  backward-render identifiability on **every** clip, shifting them off
  the solid line; the dashed line is the real "no per-clip asymmetry"
  locus. The gap between the diagonals **is** the forward bias.
- **Point colour** — `asymmetry_z_residual`: the asymmetry after an
  `arctanh` (Fisher-z) decompression of the bounded scale *and* removal
  of the forward-bias offset. Orange = the forward render carries the
  cleaner arrow-of-time cue; blue = the reversed render is the more
  conspicuous one (anti-gravity / anti-entropy events). Pale = the clip
  sits on the bias-adjusted diagonal — unremarkable.
- **Point size** — min views per direction (bigger = better sampled).
- Labelled points are the most extreme sources (greedy de-collision so
  labels don't stack).


In [ ]:
fig1 = figures.per_source_asymmetry_figure(
    save_path=PUB_DIR / 'per_source_asymmetry.pdf',
    min_views=3,      # drop sources with < 3 views in either direction
    annotate_top=8,   # label this many most-extreme sources; set 0 for none
)
fig1


## Adding more publication figures

Each figure lives as a function in `analysis/figures.py` that reads the
derived TSVs, returns a matplotlib `Figure`, and writes a PDF when given
`save_path`. To add one:

1. Write `my_new_figure(...)` in `analysis/figures.py`, applying the
   shared `_styled()` rcParams context for consistent typography.
2. Register it in the `_FIGURES` dict at the bottom of that file so
   `python -m analysis.figures` regenerates it alongside the others.
3. Add a driving cell here, mirroring Figure 1.

`python -m analysis.figures` rebuilds **every** registered figure as a
PDF in one shot — handy after a fresh pipeline run.
